# Denver Geospatial Pipeline — Visualization

This notebook visualizes the Denver study area using:
- **Google Earth Engine (GEE)** — Sentinel-2 true color & NDVI composites
- **Matplotlib** — DEM hillshade, slope, nDSM raster previews
- **GeoPandas + Matplotlib** — OSM buildings and road network

**Study area:** Denver, CO — `BBOX = (-105.05, 39.60, -104.85, 39.80)`

## 0. Setup

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
import rasterio
from rasterio.plot import show
import ee
import geemap
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

# Paths
ROOT     = os.path.abspath("..")
RAW_DIR  = os.path.join(ROOT, "data", "raw")
PROC_DIR = os.path.join(ROOT, "data", "processed")
OUT_DIR  = os.path.join(ROOT, "data", "output")

# Study area
BBOX = (-105.05, 39.60, -104.85, 39.80)  # west, south, east, north
CENTER = [39.70, -104.95]

print("Setup complete.")

## 1. Google Earth Engine — Authenticate & Initialize

In [ ]:
# Run `earthengine authenticate` in terminal once before this cell
ee.Authenticate()
ee.Initialize(project="rmtumon")  # replace with your GEE project if needed
print("GEE initialized.")

## 2. GEE — Sentinel-2 True Color & NDVI

In [ ]:
west, south, east, north = BBOX
region = ee.Geometry.BBox(west, south, east, north)

# Load Sentinel-2 SR summer composite (low cloud)
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(region)
    .filterDate("2023-06-01", "2023-09-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
    .median()
    .clip(region)
)

# NDVI band
ndvi = s2.normalizedDifference(["B8", "B4"]).rename("NDVI")

print("Sentinel-2 composite loaded.")

In [ ]:
# Interactive geemap viewer — True Color (RGB)
Map = geemap.Map(center=CENTER, zoom=12)
Map.addLayer(
    s2,
    {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000, "gamma": 1.3},
    "Sentinel-2 True Color"
)
Map.addLayer(
    ndvi,
    {"min": -0.2, "max": 0.8, "palette": ["#d73027", "#ffffbf", "#1a9641"]},
    "NDVI",
    shown=False
)
Map.add_colorbar(
    {"min": -0.2, "max": 0.8, "palette": ["#d73027", "#ffffbf", "#1a9641"]},
    label="NDVI"
)
Map

## 3. GEE — NDVI Statistics (Matplotlib)

In [ ]:
# Sample NDVI values across the region
ndvi_sample = ndvi.sample(region=region, scale=100, numPixels=500, seed=42)
ndvi_values = [f["properties"]["NDVI"] for f in ndvi_sample.getInfo()["features"]]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(ndvi_values, bins=40, color="#1a9641", edgecolor="white", linewidth=0.5)
axes[0].axvline(np.mean(ndvi_values), color="red", linestyle="--", label=f"Mean: {np.mean(ndvi_values):.3f}")
axes[0].set_xlabel("NDVI")
axes[0].set_ylabel("Pixel count")
axes[0].set_title("NDVI Distribution — Denver Summer 2023")
axes[0].legend()

# Coverage pie
bins = [-1, 0, 0.2, 0.4, 1]
labels = ["Non-veg / Water", "Bare soil", "Sparse veg", "Dense veg"]
colors = ["#4393c3", "#d6604d", "#ffffbf", "#1a9641"]
counts, _ = np.histogram(ndvi_values, bins=bins)
axes[1].pie(counts, labels=labels, colors=colors, autopct="%1.1f%%", startangle=90)
axes[1].set_title("NDVI Land Cover Classes")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "ndvi_stats.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_DIR}/ndvi_stats.png")

## 4. Matplotlib — DEM / Raster Preview

In [ ]:
raster_specs = [
    (os.path.join(PROC_DIR, "hillshade.tif"), "Hillshade",        "gray"),
    (os.path.join(PROC_DIR, "slope.tif"),     "Slope (°)",        "YlOrRd"),
    (os.path.join(PROC_DIR, "ndsm.tif"),      "nDSM Height (m)",  "viridis"),
    (os.path.join(RAW_DIR,  "dem_usgs_10m.tif"), "DEM USGS 10m", "terrain"),
]

available = [(p, t, c) for p, t, c in raster_specs if os.path.exists(p)]

if not available:
    print("No rasters found in data/processed or data/raw — run steps 02–03 first.")
else:
    fig, axes = plt.subplots(1, len(available), figsize=(6 * len(available), 5))
    if len(available) == 1:
        axes = [axes]

    for ax, (path, title, cmap) in zip(axes, available):
        with rasterio.open(path) as src:
            data = src.read(1, masked=True)
        im = ax.imshow(data, cmap=cmap, interpolation="bilinear")
        ax.set_title(title, fontsize=13, fontweight="bold")
        ax.axis("off")
        plt.colorbar(im, ax=ax, shrink=0.75, pad=0.02)

    plt.suptitle("Denver — Raster Layers", fontsize=15, y=1.02)
    plt.tight_layout()
    out = os.path.join(OUT_DIR, "raster_overview.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out}")

## 5. Matplotlib — OSM Buildings & Roads

In [ ]:
bldg_path = os.path.join(RAW_DIR, "osm_buildings.geojson")
road_path = os.path.join(RAW_DIR, "osm_roads.geojson")

# Fall back to exported output files if raw not present
if not os.path.exists(bldg_path):
    bldg_path = os.path.join(OUT_DIR, "buildings_export.geojson")
if not os.path.exists(road_path):
    road_path = os.path.join(OUT_DIR, "road_influence.geojson")

fig, ax = plt.subplots(figsize=(10, 10))
ax.set_facecolor("#1a1a2e")
fig.patch.set_facecolor("#1a1a2e")

plotted = False

if os.path.exists(road_path):
    roads = gpd.read_file(road_path)
    roads.plot(ax=ax, color="#e0e0e0", linewidth=0.4, alpha=0.6)
    print(f"Roads: {len(roads):,} features")
    plotted = True

if os.path.exists(bldg_path):
    bldgs = gpd.read_file(bldg_path)
    # Color by height if available, else flat
    height_col = next((c for c in ["lidar_height_m", "height_m", "height"] if c in bldgs.columns), None)
    if height_col:
        bldgs[height_col] = bldgs[height_col].fillna(0)
        bldgs.plot(ax=ax, column=height_col, cmap="plasma", alpha=0.85,
                   legend=True, legend_kwds={"label": "Height (m)", "shrink": 0.6})
    else:
        bldgs.plot(ax=ax, color="#e63946", alpha=0.7)
    print(f"Buildings: {len(bldgs):,} features")
    plotted = True

if not plotted:
    print("No building or road files found — run step 01 or 08 first.")
else:
    ax.set_title("Denver — OSM Buildings & Roads", color="white", fontsize=14, fontweight="bold")
    ax.tick_params(colors="white")
    for spine in ax.spines.values():
        spine.set_edgecolor("white")
    plt.tight_layout()
    out = os.path.join(OUT_DIR, "buildings_roads.png")
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    print(f"Saved: {out}")

## 6. GEE — False Color (NIR/Red/Green) Interactive Map

In [ ]:
Map2 = geemap.Map(center=CENTER, zoom=12)

# False color — vegetation shows bright red
Map2.addLayer(
    s2,
    {"bands": ["B8", "B4", "B3"], "min": 0, "max": 4000, "gamma": 1.2},
    "False Color (NIR)"
)

# SWIR composite — useful for burn scars / urban heat
Map2.addLayer(
    s2,
    {"bands": ["B11", "B8", "B4"], "min": 0, "max": 4000},
    "SWIR Composite",
    shown=False
)

Map2.addLayerControl()
Map2

## 7. Matplotlib — Monthly NDVI Time Series (GEE)

In [ ]:
months = list(range(1, 13))
mean_ndvi = []

for m in months:
    start = f"2023-{m:02d}-01"
    end   = f"2023-{m:02d}-28"
    img = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(region)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
        .median()
        .clip(region)
        .normalizedDifference(["B8", "B4"])
    )
    stats = img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=100,
        maxPixels=1e8
    ).getInfo()
    mean_ndvi.append(stats.get("nd", None))

month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]

valid = [(m, v) for m, v in zip(month_labels, mean_ndvi) if v is not None]
x_labels, y_vals = zip(*valid)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(x_labels, y_vals, marker="o", color="#1a9641", linewidth=2, markersize=7)
ax.fill_between(x_labels, y_vals, alpha=0.15, color="#1a9641")
ax.axhline(np.mean(y_vals), color="gray", linestyle="--", linewidth=1,
           label=f"Annual mean: {np.mean(y_vals):.3f}")
ax.set_title("Monthly Mean NDVI — Denver 2023 (Sentinel-2)", fontsize=13)
ax.set_ylabel("Mean NDVI")
ax.set_ylim(0, 0.6)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
out = os.path.join(OUT_DIR, "ndvi_timeseries.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")